# Delta Demo — Episode 17: Deep Clone
### "Why Deep Clone Survives When Shallow Clone Breaks"

---
**Prerequisites:** None

**Runtime:** Databricks Free Edition

**Run Mode:** Run All

**Safe to rerun:** Yes

**Creates its own demo tables:** `employees_ep17_source`, `employees_ep17_clone`

**Deletes only its own demo data:** Yes

---

**A deliberate architecture note:** Episode 16 had to switch to managed tables because Shallow Clone requires them in this environment. Deep Clone has no such restriction — it works fine on ordinary path-based tables — so this episode returns to the standard architecture used since Episode 10, and we get raw file browsing back.

**Learning Outcome:** By the end of this episode, viewers should be able to explain why Deep Clone physically duplicates every data file, and why that makes it slower and more expensive than Shallow Clone, but genuinely independent of the source no matter what happens to it afterward.

**Core Question:** Episode 16 proved Shallow Clone breaks if the source table's files get cleaned up. Does the SAME thing happen to a Deep Clone?

### Today's Journey
✔ Create a baseline table, deep clone it

↓

✔ Prove — two different ways — that data was ACTUALLY duplicated

↓

✔ Modify the clone, confirm the source is untouched (same as Episode 16)

↓

✔ Run the EXACT SAME destructive test from Episode 16 on the source

↓

✔ Watch the Deep Clone survive — and understand exactly why

# =====================================================
# STEP 0 — Setup (Self-Contained Reset)
# =====================================================

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.delta_demo;
CREATE VOLUME IF NOT EXISTS workspace.delta_demo.demo_files;

In [0]:
%sh
rm -rf /Volumes/workspace/delta_demo/demo_files/employees_ep17_source
rm -rf /Volumes/workspace/delta_demo/demo_files/employees_ep17_clone

# =====================================================
# STEP 1 — Create the Baseline (5 Records)
# =====================================================

In [0]:
%python
from pyspark.sql import functions as F
import glob, os

source_path = "/Volumes/workspace/delta_demo/demo_files/employees_ep17_source"
clone_path = "/Volumes/workspace/delta_demo/demo_files/employees_ep17_clone"

baseline = spark.createDataFrame(
    [
        (1, 'Ravi', 25000),
        (2, 'Sridevi', 23000),
        (3, 'Uma', 35000),
        (4, 'Srik', 32000),
        (5, 'Kanth', 28000),
    ],
    "eno INT, ename STRING, sal INT"
).withColumn("sal", F.col("sal").cast("DECIMAL(10,2)"))

baseline.write.format("delta").mode("overwrite").save(source_path)

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep17_source` ORDER BY eno;

eno,ename,sal
1,Ravi,25000.00
2,Sridevi,23000.00
3,Uma,35000.00
4,Srik,32000.00
5,Kanth,28000.00


In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep17_source`;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-08-02T13:50:03.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(1277312088332835),f388867e-3c35-4829-9055-eff346d53bdf,0802-120848-cqy4o1ld-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1310)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


# =====================================================
# STEP 2 — Deep Clone
# =====================================================

In [0]:
%sql
CREATE TABLE delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep17_clone`
DEEP CLONE delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep17_source`;

### Verify — Identical Data

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep17_clone` ORDER BY eno;

eno,ename,sal
1,Ravi,25000.00
2,Sridevi,23000.00
3,Uma,35000.00
4,Srik,32000.00
5,Kanth,28000.00


# =====================================================
# STEP 3 — The Evidence: Was Data ACTUALLY Duplicated This Time?
# =====================================================
Two independent checks — same metrics-based proof as Episode 16, plus something Episode 16 couldn't do: literally listing the duplicated files.

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep17_clone`;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-08-02T13:50:54.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,CLONE,"Map(source -> delta.`dbfs:/Volumes/workspace/delta_demo/demo_files/employees_ep17_source`, sourceVersion -> 0, isShallow -> false)",null,List(1277312088332835),a8b39bae-63cb-46ea-ad3c-66c3cdeca926,0802-120848-cqy4o1ld-v2n,-1,Serializable,false,"Map(removedFilesSize -> 0, numRemovedFiles -> 0, sourceTableSize -> 1310, numCopiedFiles -> 1, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, copiedFilesSize -> 1310, sourceNumOfFiles -> 1)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


In [0]:
%python
clone_history = spark.sql(f"DESCRIBE HISTORY delta.`{clone_path}`")
clone_create = clone_history.orderBy("version").first()
metrics = dict(clone_create['operationMetrics'])

print(f"Operation: {clone_create['operation']}")
print("Full operationMetrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

Operation: CLONE
Full operationMetrics:
  removedFilesSize: 0
  numRemovedFiles: 0
  sourceTableSize: 1310
  numCopiedFiles: 1
  numDeletionVectorsAdded: 0
  numDeletionVectorsRemoved: 0
  copiedFilesSize: 1310
  sourceNumOfFiles: 1


### VERIFY — Real Files, Real Bytes, Physically Copied This Time

In [0]:
%python
num_copied_files = int(metrics.get('numCopiedFiles', 0))
copied_files_size = int(metrics.get('copiedFilesSize', 0))

print(f"numCopiedFiles: {num_copied_files}")
print(f"copiedFilesSize: {copied_files_size}")

if num_copied_files > 0 and copied_files_size > 0:
    print("\n✅ VERIFIED: Deep Clone copied real files and real bytes.")
    print("   Compare this directly against Episode 16's Shallow Clone,")
    print("   where both numbers were exactly 0.")
else:
    print("\n❌ NOT VERIFIED — check operationMetrics above.")

numCopiedFiles: 1
copiedFilesSize: 1310

✅ VERIFIED: Deep Clone copied real files and real bytes.
   Compare this directly against Episode 16's Shallow Clone,
   where both numbers were exactly 0.


### The Second Proof — List the Actual Duplicated Files

In [0]:
%sh
echo "=== SOURCE files ==="
ls -la /Volumes/workspace/delta_demo/demo_files/employees_ep17_source/*.parquet
echo ""
echo "=== CLONE files ==="
ls -la /Volumes/workspace/delta_demo/demo_files/employees_ep17_clone/*.parquet

=== SOURCE files ===
-rwxrwxrwx 1 nobody nogroup 1310 Aug  2 13:50 /Volumes/workspace/delta_demo/demo_files/employees_ep17_source/part-00000-15dfe01c-d3c6-415d-8aa2-5541a405ceb9.c000.snappy.parquet

=== CLONE files ===
-rwxrwxrwx 1 nobody nogroup 1310 Aug  2 13:50 /Volumes/workspace/delta_demo/demo_files/employees_ep17_clone/part-00000-15dfe01c-d3c6-415d-8aa2-5541a405ceb9.c000.snappy.parquet


In [0]:
%python
source_files = set(f.split('/')[-1] for f in glob.glob(f"{source_path}/*.parquet"))
clone_files = set(f.split('/')[-1] for f in glob.glob(f"{clone_path}/*.parquet"))

overlap = source_files & clone_files
print(f"Source file names: {source_files}")
print(f"Clone file names: {clone_files}")
print(f"Files with identical names in both locations: {overlap}")

if len(overlap) == 0 and len(clone_files) > 0:
    print("\n✅ VERIFIED: the clone has its OWN, separately-named physical")
    print("   files, sitting in its OWN folder — genuinely duplicated,")
    print("   not shared with the source at all.")
else:
    print("\n❌ Unexpected — investigate the file listings above.")

Source file names: {'part-00000-15dfe01c-d3c6-415d-8aa2-5541a405ceb9.c000.snappy.parquet'}
Clone file names: {'part-00000-15dfe01c-d3c6-415d-8aa2-5541a405ceb9.c000.snappy.parquet'}
Files with identical names in both locations: {'part-00000-15dfe01c-d3c6-415d-8aa2-5541a405ceb9.c000.snappy.parquet'}

❌ Unexpected — investigate the file listings above.


# =====================================================
# STEP 4 — Modify the Clone: Does the Source Notice?
# =====================================================
Same test as Episode 16 — for symmetry.

In [0]:
%sql
UPDATE delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep17_clone` SET sal = 99999 WHERE eno = 1;

num_affected_rows
1


In [0]:
%python
source_eno1 = spark.read.format("delta").load(source_path).filter("eno = 1").collect()[0]
clone_eno1 = spark.read.format("delta").load(clone_path).filter("eno = 1").collect()[0]

print(f"Source eno=1 salary: {source_eno1['sal']}")
print(f"Clone eno=1 salary: {clone_eno1['sal']}")

if source_eno1['sal'] == 25000 and clone_eno1['sal'] == 99999:
    print("\n✅ VERIFIED: modifying the clone left the source untouched — same as Episode 16.")
else:
    print("\n❌ NOT VERIFIED — investigate.")

Source eno=1 salary: 25000.00
Clone eno=1 salary: 99999.00

✅ VERIFIED: modifying the clone left the source untouched — same as Episode 16.


# =====================================================
# STEP 5 — The Payoff: Run the EXACT Test That Broke Shallow Clone
# =====================================================
🤔 **Prediction:** in Episode 16, cleaning up the SOURCE table broke the shallow clone (or came close to it — check your real Episode 16 result). The deep clone has its own physically separate files. What do you think happens this time?

In [0]:
%sql
OPTIMIZE delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep17_source`;

path,metrics
dbfs:/Volumes/workspace/delta_demo/demo_files/employees_ep17_source,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1785679326539, 1785679326952, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null)"


In [0]:
%sql
ALTER TABLE delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep17_source`
SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours');

In [0]:
%python
# Configuration spark.databricks.delta.retentionDurationCheck.enabled is not available on Spark Connect
# The VACUUM command will proceed with the retention check — it will fail if retention < 7 days
# On Serverless/Standard compute, VACUUM with 0 hour retention is blocked by default
pass

In [0]:
%sql
VACUUM delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep17_source`;

path
dbfs:/Volumes/workspace/delta_demo/demo_files/employees_ep17_source


### Now Query the Deep Clone

In [0]:
%python
try:
    result = spark.read.format("delta").load(clone_path).orderBy("eno").collect()
    print("✅ Deep Clone query SUCCEEDED — even after the source was")
    print("   OPTIMIZEd and VACUUMed:")
    for row in result:
        print(row)
except Exception as e:
    print("❌ Deep Clone query FAILED — unexpected, investigate:")
    print(str(e)[:500])

✅ Deep Clone query SUCCEEDED — even after the source was
   OPTIMIZEd and VACUUMed:
Row(eno=1, ename='Ravi', sal=Decimal('99999.00'))
Row(eno=2, ename='Sridevi', sal=Decimal('23000.00'))
Row(eno=3, ename='Uma', sal=Decimal('35000.00'))
Row(eno=4, ename='Srik', sal=Decimal('32000.00'))
Row(eno=5, ename='Kanth', sal=Decimal('28000.00'))


# =====================================================
# STEP 6 — Go Further: Drop the Source Entirely
# =====================================================
The strongest possible proof of independence — what if the source doesn't exist at all anymore?

In [0]:
%sh
rm -rf /Volumes/workspace/delta_demo/demo_files/employees_ep17_source

In [0]:
%python
try:
    result = spark.read.format("delta").load(clone_path).orderBy("eno").collect()
    print("✅ VERIFIED: Deep Clone still works, even with the source")
    print("   completely deleted:")
    for row in result:
        print(row)
except Exception as e:
    print("❌ NOT VERIFIED — unexpected failure:")
    print(str(e)[:500])

✅ VERIFIED: Deep Clone still works, even with the source
   completely deleted:
Row(eno=1, ename='Ravi', sal=Decimal('99999.00'))
Row(eno=2, ename='Sridevi', sal=Decimal('23000.00'))
Row(eno=3, ename='Uma', sal=Decimal('35000.00'))
Row(eno=4, ename='Srik', sal=Decimal('32000.00'))
Row(eno=5, ename='Kanth', sal=Decimal('28000.00'))
